# Notebook 01: RAG — Ask Questions About Scientific Literature

**CABS AI Productivity Series**  
Workshop: *LangChain, LangGraph & Local LLM Deployment: Building AI Agent Systems That Keep Your Data Safe*

---

## What is RAG?

**RAG (Retrieval-Augmented Generation)** lets an LLM answer questions based on *your* documents, not just its training data.

### How it works:

Your documents → Split into chunks → Convert to embeddings → Store in vector database

User question → Convert to embedding → Find most relevant chunks → LLM reads chunks + question → Answer

### What you'll build in this notebook:

A system that reads 10 PubMed abstracts about Organ-on-a-chip, and answers your questions based on what those papers actually say.

### Why this matters for biologists:

- You can ask questions across multiple papers at once
- The LLM only answers from your provided documents, reducing hallucination
- This is the foundation of "chat with your data" tools

---
## Setup: Get Your Free Gemini API Key

1. Go to [aistudio.google.com](https://aistudio.google.com)
2. Sign in with your Google account
3. Accept Terms of Service
4. Left sidebar → **Get API Key** → **Create API key**
5. Copy the key — it starts with `AIza...`

Free tier should be sufficient for this demo. No credit card needed. Run the two cells below to install packages and enter your key.

In [ ]:
# ============================================================
# STEP 1: Install required packages
# ============================================================
# langchain              - framework for building LLM applications
# langchain-google-genai - connects LangChain to Google's Gemini models
# langchain-text-splitters - tools to split documents into chunks
# chromadb               - lightweight vector database, stores embeddings in memory

!pip install -q langchain langchain-google-genai langchain-text-splitters chromadb langchain-community

In [ ]:
# ============================================================
# STEP 2: Enter your Gemini API key
# ============================================================
# getpass hides your input so the key won't show on screen
#
# IMPORTANT: Never hardcode API keys in notebooks you share!

import getpass
import os

api_key = getpass.getpass("Paste your Gemini API key here: ")
os.environ["GOOGLE_API_KEY"] = api_key

print("✅ API key set!")

In [ ]:
# ============================================================
# STEP 3: Quick test — make sure your API key works
# ============================================================

from langchain_google_genai import ChatGoogleGenerativeAI

# Initialize the Gemini model
# gemini-2.5-flash is fast and free-tier friendly
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash")

# Send a simple test message
response = llm.invoke("Say hello in one sentence.")
print(response.content)
print("\n✅ Gemini is working!")

---
## Prepare the Documents

Below are 5 PubMed abstracts about **Organ-on-a-chip methods**. In a real project, you could load these from PDF files or a database. Here we include them directly for simplicity.

In [ ]:
# ============================================================
# STEP 4: Load our documents (10 organ-on-a-chip abstracts)
# ============================================================
# These abstracts are written for this demo based on real research
# themes in the organ-on-a-chip field. In a real project, you would
# load actual papers from PubMed, PDF files, or a reference manager.
#
# Topics covered:
#   1. Lung-on-a-chip (respiratory modeling)
#   2. Gut-on-a-chip (microbiome co-culture)
#   3. Liver-on-a-chip (drug metabolism / hepatotoxicity)
#   4. Blood-brain barrier on chip
#   5. Heart-on-a-chip (cardiotoxicity)
#   6. Kidney-on-a-chip (nephrotoxicity)
#   7. Tumor-on-a-chip (cancer microenvironment)
#   8. Multi-organ-on-a-chip (systemic drug testing)
#   9. Gut-brain axis on chip
#   10. Organ-on-chip for personalized medicine (patient-derived cells)

from langchain_core.documents import Document

documents = [
    Document(
        page_content="""
        A human lung-on-a-chip device was developed to recapitulate the alveolar-
        capillary interface by co-culturing primary human alveolar epithelial cells
        and pulmonary microvascular endothelial cells on opposite sides of a thin,
        porous PDMS membrane. Cyclic mechanical strain was applied to mimic
        breathing motions. The device successfully reproduced inflammatory responses
        to bacterial infection and cytokine stimulation, including neutrophil
        recruitment from the vascular channel. Drug testing with a known anti-
        inflammatory compound showed dose-dependent reduction of IL-8 secretion,
        results that closely matched clinical observations but were not predicted
        by static cell culture models. The platform demonstrates the critical role
        of mechanical forces in pulmonary drug responses.
        """,
        metadata={"source": "Paper_01", "title": "Lung-on-a-chip with breathing motions", "year": 2023, "organ": "lung"}
    ),
    Document(
        page_content="""
        This study presents a gut-on-a-chip platform incorporating a co-culture
        of human intestinal epithelial cells (Caco-2) and commensal gut bacteria
        (Lactobacillus rhamnosus) under continuous flow conditions. The microfluidic
        device maintained stable bacterial populations for over 72 hours without
        compromising epithelial barrier integrity, a result not achievable in
        conventional Transwell systems where bacterial overgrowth kills host cells
        within 24 hours. Transcriptomic analysis revealed that bacterial co-culture
        under flow conditions upregulated mucin production and tight junction
        proteins compared to bacteria-free controls. The platform was used to
        study how antibiotic treatment disrupts barrier function and enables
        pathogenic E. coli translocation across the epithelium.
        """,
        metadata={"source": "Paper_02", "title": "Gut-on-a-chip with microbiome co-culture", "year": 2023, "organ": "gut"}
    ),
    Document(
        page_content="""
        An advanced liver-on-a-chip model was constructed using primary human
        hepatocytes, stellate cells, and Kupffer cells arranged in a 3D
        configuration within microfluidic channels. The tri-culture system
        maintained cytochrome P450 enzyme activity for 28 days, significantly
        exceeding the 5-7 day functional window of conventional 2D hepatocyte
        cultures. Drug metabolism studies with acetaminophen demonstrated dose-
        dependent hepatotoxicity with an IC50 value within two-fold of the known
        human toxic concentration, whereas standard in vitro assays overestimated
        the toxic dose by 10-fold. The model also detected metabolite-mediated
        toxicity from cyclophosphamide, a prodrug requiring hepatic activation,
        which static culture systems fail to capture due to rapid loss of
        metabolic competence.
        """,
        metadata={"source": "Paper_03", "title": "Liver-on-a-chip for drug metabolism and hepatotoxicity", "year": 2024, "organ": "liver"}
    ),
    Document(
        page_content="""
        A microfluidic blood-brain barrier (BBB) chip was engineered using human
        induced pluripotent stem cell-derived brain microvascular endothelial cells
        co-cultured with astrocytes and pericytes. The device achieved
        transendothelial electrical resistance (TEER) values exceeding
        2000 ohm-cm2, approaching physiological levels and far surpassing
        conventional Transwell BBB models (typically 200-400 ohm-cm2).
        Permeability assays with 12 known CNS drugs demonstrated strong
        correlation (R2 = 0.89) with in vivo brain penetration data. The chip
        was used to model neuroinflammation by introducing TNF-alpha, which
        caused reversible barrier disruption and increased permeability to
        large-molecule therapeutics. This platform enables screening of drug
        candidates for CNS delivery and neurotoxicity assessment.
        """,
        metadata={"source": "Paper_04", "title": "Blood-brain barrier on chip using iPSC-derived cells", "year": 2023, "organ": "brain"}
    ),
    Document(
        page_content="""
        A heart-on-a-chip platform was developed using human iPSC-derived
        cardiomyocytes cultured on flexible thin-film cantilevers within a
        microfluidic device. The system continuously monitored contractile
        force, beating frequency, and calcium transients in real time. Testing
        with 15 drugs known to cause clinical cardiotoxicity demonstrated that
        the chip correctly identified 13 of 15 cardiotoxic compounds, including
        drugs that passed preclinical animal testing but failed in human clinical
        trials due to QT prolongation. The platform detected both acute effects
        (within hours) and chronic toxicity patterns (over 14 days of repeated
        dosing). Integration of real-time electrical impedance sensing eliminated
        the need for fluorescent reporters, enabling non-invasive longitudinal
        monitoring.
        """,
        metadata={"source": "Paper_05", "title": "Heart-on-a-chip for cardiotoxicity screening", "year": 2024, "organ": "heart"}
    ),
    Document(
        page_content="""
        A proximal tubule kidney-on-a-chip was fabricated using primary human
        renal proximal tubular epithelial cells cultured under physiological
        fluid shear stress. Cells formed polarized monolayers with functional
        brush borders and expressed drug transporters (OAT1, OAT3, OCT2) at
        levels comparable to freshly isolated tissue, unlike static cultures
        where transporter expression declines rapidly. Nephrotoxicity testing
        with cisplatin revealed dose-dependent biomarker release (KIM-1, NGAL)
        that preceded cell death by 48 hours, providing early warning signals
        not captured by viability assays alone. The device also demonstrated
        active tubular secretion and reabsorption of organic solutes, enabling
        quantitative prediction of renal drug clearance that correlated with
        clinical pharmacokinetic data.
        """,
        metadata={"source": "Paper_06", "title": "Kidney-on-a-chip for nephrotoxicity and drug clearance", "year": 2023, "organ": "kidney"}
    ),
    Document(
        page_content="""
        A vascularized tumor-on-a-chip was developed to study the tumor
        microenvironment and drug transport. Patient-derived glioblastoma
        spheroids were embedded in a collagen matrix and perfused through
        adjacent endothelialized microchannels that mimicked tumor vasculature.
        The device reproduced key features of the in vivo tumor microenvironment,
        including hypoxic gradients, angiogenic sprouting, and immune cell
        infiltration. Drug testing with temozolomide showed that the
        vascularized chip model predicted clinical response rates more
        accurately than standard spheroid assays, capturing the limited drug
        penetration through tumor vasculature that contributes to treatment
        resistance. The platform was further used to evaluate combination
        therapies with anti-angiogenic agents that are difficult to test in
        conventional culture systems.
        """,
        metadata={"source": "Paper_07", "title": "Vascularized tumor-on-a-chip for drug penetration studies", "year": 2024, "organ": "tumor"}
    ),
    Document(
        page_content="""
        A multi-organ-on-a-chip system connecting liver, kidney, heart, and
        intestine compartments through a common microfluidic circulation was
        developed for systemic drug toxicity assessment. Each organ compartment
        contained tissue-specific human cell types cultured under physiologically
        scaled flow rates and media volumes. The interconnected system successfully
        demonstrated first-pass metabolism of the prodrug terfenadine in the
        liver compartment, followed by detection of its cardiotoxic metabolite
        in the heart compartment — an inter-organ toxicity mechanism that cannot
        be captured in single-organ models. Pharmacokinetic modeling of drug
        distribution across compartments showed absorption, distribution,
        metabolism, and excretion (ADME) profiles consistent with human clinical
        data, validating the platform as a preclinical tool for predicting
        systemic drug behavior.
        """,
        metadata={"source": "Paper_08", "title": "Multi-organ-on-a-chip for systemic ADME profiling", "year": 2024, "organ": "multi-organ"}
    ),
    Document(
        page_content="""
        A gut-brain axis chip was constructed by fluidically linking a gut
        compartment containing intestinal epithelium and commensal bacteria
        to a brain compartment containing neurons and astrocytes, connected
        through a blood-brain barrier module. The system demonstrated that
        microbial metabolites, specifically short-chain fatty acids produced
        by bacterial fermentation in the gut module, could cross the BBB
        module and modulate neuronal calcium signaling in the brain compartment.
        Conversely, inflammatory cytokines introduced in the brain module
        altered gut barrier permeability through the circulatory connection.
        This bidirectional communication was abolished when the BBB module
        was removed, confirming the barrier's gatekeeping role. The platform
        provides a controlled environment to study microbiome-brain
        interactions implicated in neurodegenerative diseases.
        """,
        metadata={"source": "Paper_09", "title": "Gut-brain axis on chip with BBB module", "year": 2024, "organ": "gut-brain"}
    ),
    Document(
        page_content="""
        This study demonstrates the use of patient-derived tumor organoids
        integrated into organ-on-a-chip devices for personalized drug
        sensitivity testing. Tumor biopsies from 12 colorectal cancer
        patients were expanded as organoids and loaded into standardized
        microfluidic chips with perfused vasculature. A panel of 8
        chemotherapy regimens was tested on each patient's chip, with
        results available within 7 days of biopsy. Drug sensitivity
        profiles varied substantially across patients, and retrospective
        comparison with clinical outcomes showed 83% concordance between
        chip predictions and actual patient responses. The platform
        identified effective drug combinations for two patients whose
        tumors were resistant to standard first-line therapy. Scalable
        chip fabrication and automated imaging analysis enable potential
        clinical deployment for treatment selection.
        """,
        metadata={"source": "Paper_10", "title": "Patient-derived organoid chips for personalized oncology", "year": 2024, "organ": "tumor-personalized"}
    ),
]

print(f"✅ Loaded {len(documents)} documents")
print()

# Show all document titles
for i, doc in enumerate(documents):
    print(f"  {i+1}. [{doc.metadata['organ']}] {doc.metadata['title']} ({doc.metadata['year']})")

---
## Split Documents into Chunks

Even though our abstracts are short, in real projects your documents could be full papers (10+ pages). Splitting into smaller chunks ensures the retriever can find the most relevant *piece* of text, not just the most relevant *paper*.

Key parameters:
- **chunk_size**: How long each chunk is (in characters)
- **chunk_overlap**: How much adjacent chunks overlap, to avoid cutting sentences in half

In [ ]:
# ============================================================
# STEP 5: Split documents into chunks
# ============================================================

from langchain_text_splitters import RecursiveCharacterTextSplitter

# RecursiveCharacterTextSplitter tries to split at natural boundaries:
# first paragraphs, then sentences, then words

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,       # each chunk is at most 500 characters
    chunk_overlap=50,     # adjacent chunks share 50 characters
    length_function=len,
)

# Split all documents
chunks = text_splitter.split_documents(documents)

print(f"✅ Split {len(documents)} documents into {len(chunks)} chunks")
print()
print("--- Example chunk ---")
print(f"Source: {chunks[0].metadata['source']}")
print(f"Content: {chunks[0].page_content.strip()[:300]}")

 ---
## Create Embeddings & Vector Store

**Embeddings** convert text into numbers (vectors) that capture meaning. Similar texts produce similar vectors. This is how the system finds relevant paper chunks when you ask a question.

Example:

"liver-on-a-chip hepatotoxicity drug metabolism" → [0.12, -0.34, 0.56, ...]

"microfluidic liver model for testing drug safety" → [0.11, -0.32, 0.55, ...] ← very similar!

"Python programming for beginners" → [0.87, 0.21, -0.65, ...] ← very different!

Notice that the first two texts use completely different words but mean similar things. Embeddings capture this semantic similarity — which is why you can ask "which chip detects kidney damage" and still retrieve a paper that says "nephrotoxicity biomarker release in proximal tubule device."

In [ ]:
# ============================================================
# STEP 6: Create embeddings and store in vector database
# ============================================================

from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_community.vectorstores import Chroma

# Create the embedding model
# This model converts text → vectors
embedding_model = GoogleGenerativeAIEmbeddings(
    model="models/gemini-embedding-001"
)
# Create vector store from our chunks
# Chroma is a lightweight vector database that runs in memory
#
# What happens here:
#   1. Each chunk's text is sent to Google's embedding API → gets a vector back
#   2. The vector + original text + metadata are stored in Chroma

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embedding_model,
)

print(f"✅ Vector store created with {vectorstore._collection.count()} chunks")

In [ ]:
# ============================================================
# STEP 7: Test the retriever — see what comes back for a query
# ============================================================
# Before building the full RAG chain, let's see the retrieval step alone.
# This helps you understand what the LLM will "see" when answering.

# Create a retriever that returns the top 3 most relevant chunks
retriever = vectorstore.as_retriever(
    search_kwargs={"k": 3}  # k = number of chunks to retrieve
)

# Test query
test_query = "How does the gut chip keep bacteria alive without killing host cells?"
retrieved_docs = retriever.invoke(test_query)

print(f"Query: '{test_query}'")
print(f"\nRetrieved {len(retrieved_docs)} relevant chunks:\n")

for i, doc in enumerate(retrieved_docs):
    print(f"--- Chunk {i+1} (from {doc.metadata['source']}) ---")
    print(doc.page_content.strip()[:200])
    print()

---
## Build the RAG Chain

Now we connect everything: **retriever** (finds relevant chunks) + **LLM** (reads them and answers).

This is the core pattern. The LLM doesn't search the internet — it only reads the chunks you retrieved. This is what makes RAG controllable.

In [ ]:
# ============================================================
# STEP 8: Build the RAG chain
# ============================================================

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

# The prompt template tells the LLM how to behave
# Key instruction: "Only answer based on the provided context"
# This prevents the LLM from making things up!

prompt_template = ChatPromptTemplate.from_template("""
You are a helpful research assistant. Answer the question based ONLY on the
following context from scientific papers. If the context doesn't contain
enough information to answer, say "I don't have enough information in the
provided papers to answer this."

Always cite which paper(s) you're drawing from using their PMID.

Context from papers:
{context}

Question: {question}

Answer:
""")


# Helper function: format retrieved docs into a single string
def format_docs(docs):
    formatted = []
    for doc in docs:
        source = doc.metadata.get('source', 'Unknown')
        title = doc.metadata.get('title', 'Unknown')
        formatted.append(f"[{source} - {title}]\n{doc.page_content.strip()}")
    return "\n\n".join(formatted)


# Build the chain using LangChain Expression Language (LCEL)
# Flow: question → retriever → format chunks → plug into prompt → send to LLM → answer

rag_chain = (
    {
        "context": retriever | format_docs,   # retrieve & format
        "question": RunnablePassthrough()      # pass the question through as-is
    }
    | prompt_template   # fill in the template
    | llm               # send to Gemini
    | StrOutputParser()  # extract the text response
)

print("✅ RAG chain built!")
print("Ready to answer questions from your literature.")

---
## Ask Questions!

Now you can ask questions about the 5 papers **in natural language**. The system will:
1. Convert your question to a vector
2. Find the most relevant paper chunks
3. Have the LLM read those chunks and write an answer

This is exactly how real "chat with your documents" tools work — the same pattern, just with more documents.

In [ ]:
# ============================================================
# STEP 9: Ask your first question
# ============================================================

question = "What cardiotoxic drugs did the heart chip catch that animal testing missed?"

print(f"❓ Question: {question}")
print("\n" + "="*60)
print("Searching papers and generating answer...")
print("="*60 + "\n")

answer = rag_chain.invoke(question)
print(answer)

In [ ]:
# ============================================================
# STEP 10: More example questions — try them all!
# ============================================================
# These show different types of questions you can ask.
# Change the number (0-4) to try a different one, or write your own!

example_questions = [
    # Comparison question
    "Which organ chips use iPSC-derived cells?",

    # Specific detail question
    "If I want to test whether my drug causes liver or kidney damage, which platform should I use?",

    # Practical decision question
    "What are the main advantages of organ-on-a-chip over traditional cell culture?",

    # Challenge question: ask about something NOT in the papers
    "How much does it cost to fabricate an organ-on-a-chip?",

    # Synthesis question
    "What regulatory guidelines has the FDA issued for organ-on-a-chip data?",
]

# Pick one (change the number 0-4)
question = example_questions[4]

print(f"❓ Question: {question}\n")
answer = rag_chain.invoke(question)
print(answer)

In [ ]:
# ============================================================
# STEP 11: Try your own question!
# ============================================================
# Type any question about organ-on-a-chip below.
# The system will search the 5 papers and answer.
#
# Try asking in different ways — notice how the same information
# can be retrieved with very different phrasings!

my_question = "Which chip would be most useful for studying how antibiotics affect the gut?"

print(f"❓ Your question: {my_question}\n")
answer = rag_chain.invoke(my_question)
print(answer)

---
## What Just Happened?

Full flow recap:

Your question (natural language) → Converted to embedding vector → Vector search finds top 3 most similar chunks → Chunks + question inserted into prompt template → Sent to Gemini LLM (cloud API) → Gemini reads the chunks and writes an answer

### Key takeaways:

1. **The LLM only sees what you give it.** It answered from your 5 papers, not from the internet.

2. **This is controllable.** You decide which documents go in, how they're chunked, how many chunks are retrieved.

3. **This scales.** The same pattern works with 5 papers or 5,000 papers.

4. **But — your data went to Google's server.** Every question and every chunk was sent to the Gemini API for processing. For public literature, this is fine. For unpublished data or patient data, this matters.

---

### What's next?

**Notebook 02** shows you how to go beyond just reading documents. What if you want the LLM to **take action** — like looking up a gene in an external database? That's an **Agent**.

**Notebook 03** addresses data privacy: running the LLM **locally on your own machine** with Ollama, so your documents and questions never leave your computer.

👉 Continue to [02_gene_lookup_agent.ipynb](https://github.com/lecaibio/cabs-workshop-llm-agents/blob/main/notebooks/02_gene_lookup_agent.ipynb)